In [1]:
import pandas as pd
from openai import OpenAI
from sklearn.metrics import accuracy_score

df = pd.read_csv("final_train_data.csv").sample(50, random_state=42)

def create_ultimate_prompt(row):
    # modele rehberlik etmesi için 2 gercek senaryoyu kurallarla veriyoruz
    return f"""
    Sen bir kaza analiz uzmanısın. Yolcuların verilerini inceleyerek transfer olasılığını hesapla.

    BİLGİ REHBERİ:
    - ÖRNEK 1: Kriyojenik uykuda olan (True) ve harcaması 0 olan yolcuların transfer olma olasılığı çok yüksektir. (SONUÇ: TRUE)
    - ÖRNEK 2: Uyanık olan (False), Spa ve VRDeck'te binlerce birim harcayan yolcuların transfer olma olasılığı düşüktür. (SONUÇ: FALSE)

    YENİ ANALİZ EDİLECEK YOLCU:
    - Kriyojenik Uyku: {'Evet' if row.get('CryoSleep') else 'Hayır'}
    - Memleket: {row.get('HomePlanet')}
    - Yaş: {row.get('Age')}
    - Toplam Harcama (Spa, VR, Oda Servisi): {row.get('RoomService', 0) + row.get('Spa', 0) + row.get('VRDeck', 0)}

    Analiz: Bu yolcunun durumunu yukarıdaki uzman kuralları ve örneklerle kıyaslayarak bir karara var.
    Lütfen SADECE en sonda 'SONUÇ: TRUE' veya 'SONUÇ: FALSE' yaz.
    """

df['Ultimate_Prompt'] = df.apply(create_ultimate_prompt, axis=1)
client = OpenAI(api_key="api-key")

# karsilastirilacak modeller
modeller = ["gpt-3.5-turbo", "gpt-4"]

for model_adi in modeller:
    tahminler = []
    print(f"{model_adi} (Uzman + Few-Shot) test ediliyor...")

    for prompt in df['Ultimate_Prompt']:
        try:
            response = client.chat.completions.create(
                model=model_adi,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            cevap = response.choices[0].message.content.strip().upper()

            if "TRUE" in cevap: tahminler.append(True)
            elif "FALSE" in cevap: tahminler.append(False)
            else: tahminler.append(None)
        except:
            tahminler.append(None)

    col_name = f'{model_adi}_Expert_Prediction'
    df[col_name] = tahminler
    temiz = df.dropna(subset=[col_name, 'Transported'])
    acc = accuracy_score(temiz['Transported'].astype(bool), temiz[col_name].astype(bool))

    gosterim_adi = "GPT-3.5" if "3.5" in model_adi else "GPT-4"
    print(f"--- {gosterim_adi} Yeni Uzman Skor: %{acc * 100:.2f}")


gpt-3.5-turbo (Uzman + Few-Shot) test ediliyor...


ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from scipy.optimize import differential_evolution

# ornek veri
df = pd.read_csv("submission_kesin_cozum.csv")

df = df.astype(int)
y_true = df['Transported']

# senaryo gruplari
trad_models = ['XGBoost', 'LightGBM', 'RandomForest']
all_models = ['XGBoost', 'LightGBM', 'RandomForest', 'GPT4', 'GPT35']

# metod 1: klasik oylama
def majority_voting(predictions_df):
    # satır bazında toplamı alip, model sayısinin yarisindan buyukse 1, degilse 0 yap
    return (predictions_df.sum(axis=1) > (predictions_df.shape[1] / 2)).astype(int)

# metod 2: evrimsel hesaplama ile optimizasyon
def optimize_weights(predictions_df, y_true):
    preds_matrix = predictions_df.values
    n_models = preds_matrix.shape[1]

    # optimizasyon için kayip fonksiyonu: 1 - accuracy
    def loss_function(weights):
        # agirlikli oylama
        weighted_sum = np.dot(preds_matrix, weights)
        threshold = np.sum(weights) / 2
        ensemble_pred = (weighted_sum > threshold).astype(int)
        return 1.0 - accuracy_score(y_true, ensemble_pred)

    # evrimsel algoritma ayarlari (her modelin agirligi 0 ile 1 arasinda olabilir)
    bounds = [(0.0, 1.0) for _ in range(n_models)]

    # meta-sezgisel algoritma
    result = differential_evolution(loss_function, bounds, strategy='best1bin', popsize=15, mutation=(0.5, 1), recombination=0.7, seed=42)

    best_weights = result.x
    best_accuracy = 1.0 - result.fun
    return best_weights, best_accuracy

print("Senaryo 1: Sadece Geleneksel Modeller")
trad_voting_pred = majority_voting(df[trad_models])
print(f"1. Klasik Oylama Dogrulugu: %{accuracy_score(y_true, trad_voting_pred)*100:.2f}")

trad_weights, trad_evo_acc = optimize_weights(df[trad_models], y_true)
print(f"2. Evrimsel Hesaplama Dogrulugu: %{trad_evo_acc*100:.2f}")
print(f"   Bulunan Optimum Agirliklar: XGBoost({trad_weights[0]:.2f}), LightGBM({trad_weights[1]:.2f}), RF({trad_weights[2]:.2f})\n")

print("Senaryo 2: Geleneksel Modeller + LLM'ler")
all_voting_pred = majority_voting(df[all_models])
print(f"1. Klasik Oylama Dogrulugu: %{accuracy_score(y_true, all_voting_pred)*100:.2f}")

all_weights, all_evo_acc = optimize_weights(df[all_models], y_true)
print(f"2. Evrimsel Hesaplama Dogrulugu: %{all_evo_acc*100:.2f}")
print(f"   Bulunan Optimum Agirliklar: XGB({all_weights[0]:.2f}), LGBM({all_weights[1]:.2f}), RF({all_weights[2]:.2f}), GPT4({all_weights[3]:.2f}), GPT35({all_weights[4]:.2f})")